In [5]:
# import os
# pistack_dir = os.getcwd()
# os.chdir('../')
# from edautils.plot import ternary_ax, ternary
# from edautils.plot import calc_portion, energy_bar_ax
# from edautils.plot import kdeplot, kdeplot_norm
# from edautils.plot import violin
# from edautils.plot import plot
import sys, os
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from edautils.fragment import find_rings_old
from rdkit import Chem
from rdkit.Chem import rdmolops, AllChem, Draw
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [20]:


hetratom_prot = []
aromatic_prot = []

processed_df = pd.read_csv('pi_stack_EDA.csv', index_col=0)
# print('# Total pairs:', len(df))
# print('# Repulsive pairs:', len(df.query('TOTAL > 0')))
# print('# Attractive pairs:', len(df.query('TOTAL < 0')))

for index, row in processed_df.iterrows():
    if row['restype'] == 'TYR' or row['restype'] == 'PHE':
        aromatic_prot.append(index)
    else:
        hetratom_prot.append(index)

# Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in set(df['smiles1'])], subImgSize=(300, 300), molsPerRow=5)

dataset_dir = '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset'
hetratom_lig = []
aromatic_lig = []

hetrato_atoms = []
aromatic_atoms = []

errs = []

for index, row in processed_df.iterrows():
    aromatic = True
    temp_atomlist = []
    ligand = os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/{row['full_PDBID']}_ligand_refined.sdf")
    mol = Chem.SDMolSupplier(ligand, removeHs=False, sanitize=True)[0]
    ring_list = find_rings_old(mol, find_hetero_rings=False, find_small_rings=False) # only aromatic rings, idx numbers in OpenBabel 1 lower than Plip

    lig_idx_list = row['lig_idx_list'].split()
    lig_idx_set = set([int(id) - 1 for id in lig_idx_list])
    ring_temp = [s for s in ring_list if lig_idx_set & s]

    if row['PDBID'] == '3b2q': # Plip ring missidentification
        ring_temp = [{21, 22, 23, 24, 25, 27, 28, 29, 30}]
    elif row['PDBID'] == '2cm8': # reassign to aromatic ring
        ring_temp = [{8, 9, 10, 11, 12, 13}] 

    try:
        for idx in ring_temp[0]:
            atom = mol.GetAtomWithIdx(idx)
            symbol = atom.GetSymbol()
            temp_atomlist.append(symbol)
            if not symbol == 'C':
                aromatic = False

        if aromatic:
            aromatic_lig.append(index)
            aromatic_atoms.append((index, temp_atomlist)) # keep the list of atoms
        else:
            hetratom_lig.append(index)
            hetrato_atoms.append((index, temp_atomlist)) # keep the list of atoms
    except Exception as e:
        print(index, lig_idx_set, ring_list)
        # raise e
        for idx in lig_idx_set:
            atom = mol.GetAtomWithIdx(idx)
            symbol = atom.GetSymbol()
            temp_atomlist.append(symbol)
        errs.append([index, temp_atomlist])

errPDBID = [err[0] for err in errs]
print(len(errPDBID))

processed_df["group"] = pd.NA

for index, row in processed_df.iterrows():
    if index in errPDBID:
        continue
    if index in hetratom_prot:
        processed_df.loc[index, "prot_hetero"] = True
    else:
        processed_df.loc[index, "prot_hetero"] = False
    
    if index in hetratom_lig:
        processed_df.loc[index, "lig_hetero"] = True
    else:
        processed_df.loc[index, "lig_hetero"] = False    
    
    if index in hetratom_prot and index in hetratom_lig:
        processed_df.loc[index, "group"] = "Heterocyclic"
    elif index in aromatic_lig and index in aromatic_lig:
        processed_df.loc[index, "group"] = "Carbocyclic"
    else:
        processed_df.loc[index, "group"] = "Mixed"

processed_df.to_csv('pi_stack_EDA.csv')

3atw_42_HIS_692-698_23-28 {22, 23, 24, 25, 26, 27} [{32, 33, 34, 35, 31}]
2hwi_478_TYR_7442-7451_13-17 {12, 13, 14, 15, 16} [{0, 1, 2, 3, 4, 5}, {19, 20, 21, 22, 23}]
1o86_485_TYR_7828-7837_24-28 {23, 24, 25, 26, 27} [{6, 7, 8, 9, 10, 11}]
6dgr_244_HIS_3965-3972_4-22 {3, 4, 15, 16, 21} [{0, 1, 11, 12, 13, 14}, {20, 5, 6, 7, 8}]
6dgl_59_PHE_978-987_8-30 {7, 8, 22, 24, 29} [{0, 16, 17, 13, 14, 15}, {3, 4, 5, 6, 19, 20}, {23, 25, 10, 11, 12}]
2hha_629_TYR_10134-10143_4-16 {3, 4, 5, 6, 15} [{20, 9, 10, 11, 12, 13}, {17, 18, 19, 7, 8}]
3wtm_186_TYR_2926-2935_7-14 {6, 9, 11, 12, 13} [{1, 2, 3, 4, 7, 8}]
5oh7_166_TRP_2563-2573_2-6 {1, 2, 3, 4, 5} []
4bj8_212_TRP_3179-3192_11-16 {10, 11, 12, 14, 15} []
2ovh_99_PHE_1649-1658_11-17 {10, 11, 12, 14, 15, 16} [{22, 23, 24, 25, 29, 30}]
5oha_69_TRP_1065-1071_2-6 {1, 2, 3, 4, 5} []
2bgd_183_PHE_3049-3058_2-8 {1, 2, 3, 6, 7} [{8, 9, 10, 11, 12, 13}, {16, 17, 18, 19, 20, 21}]
3wth_55_TRP_896-906_9-13 {8, 9, 10, 11, 12} [{0, 1, 2, 4, 5, 6}]
5ov9_285_HIS

,PDBID,full_PDBID,subdir,resnr,restype,reschain,resnr_lig,restype_lig,reschain_lig,prot_idx_list,...,MOD_PAULI,CLS_DISP,PREPARATION,FROZEN,POLARIZATION,CHARGE_TRANSFER,TOTAL,group,prot_hetero,lig_hetero
3t19_231_TRP_3830-3843_12-17,3t19,3t19_5MA_A_561,pdbbind_opt_sm/,231,TRP,A,1,UNL,Z,3830 3831 3833 3834 3836 3837 3839 3841 3843,...,29.3947,NaN,0.0,-11.5138,-2.6237,-3.7063,-17.8438,Carbocyclic,True,False
5lkr_757_TRP_11846-11859_2-7,5lkr,5lkr_6YC_B_611,pdbbind_opt_sm/,757,TRP,B,1,UNL,Z,11846 11847 11849 11850 11852 11853 11855 1185...,...,12.5846,NaN,-0.0,-9.4504,-1.3915,-2.1749,-13.0167,Carbocyclic,True,False
2oqi_629_TYR_10130-10139_7-27,2oqi,2oqi_GGO_B_901,pdbbind_opt_sm/,629,TYR,B,1,UNL,Z,10130 10131 10133 10135 10137 10139,...,11.5765,NaN,0.0,-6.6099,-1.3825,-1.8063,-9.7987,Carbocyclic,False,False
3r6t_142_TRP_2266-2279_28-32,3r6t,3r6t_LU1_A_307,pdbbind_opt_sm/,142,TRP,A,1,UNL,Z,2266 2267 2269 2270 2272 2273 2275 2277 2279,...,45.9564,NaN,0.0,-0.5732,-4.8675,-4.7810,-10.2217,Heterocyclic,True,True
4pg3_746_HIS_12207-12214_3-9,4pg3,4pg3_KRS_D_602,pdbbind_opt_sm/,746,HIS,D,1,UNL,Z,12207 12208 12210 12212 12214,...,32.4816,NaN,0.0,1.4604,-2.1902,-4.3733,-5.1030,Carbocyclic,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6h12_119_TYR_1926-1935_15-46,6h12,6h12_FJK_B_619,pdbbind_opt_sm/,119,TYR,B,1,UNL,Z,1926 1927 1929 1931 1933 1935,...,76.9962,NaN,-0.0,17.9936,-12.5863,-13.2766,-7.8693,Mixed,False,True
3uuo_581_PHE_9405-9410_1-20,3uuo,3uuo_0CV_B_1,pdbbind_opt_sm/,581,PHE,B,1,UNL,Z,9405 9406 9407 9408 9409 9410,...,30.8768,NaN,0.0,-19.0010,-1.9611,-3.6242,-24.5863,Mixed,False,True
4zph_507_TRP_8199-8205_5-14,4zph,4zph_PRL_A_501,pdbbind_opt_sm/,507,TRP,B,1,UNL,Z,8199 8200 8202 8203 8205,...,4.8568,NaN,-0.0,-20.0494,-1.3352,-1.4919,-22.8766,Heterocyclic,True,True
2vvu_206_TRP_3228-3241_5-10,2vvu,2vvu_H22_A_1244,pdbbind_opt_sm/,206,TRP,A,1,UNL,Z,3228 3229 3231 3232 3234 3235 3237 3239 3241,...,37.3615,NaN,0.0,-18.5353,-4.6194,-5.8222,-28.9768,Heterocyclic,True,True
